# Bai recovery: rebuild first-candidate recipe + corrected re-evaluation

The saved Kaggle Version 1 output contains no trained adapter, so direct re-evaluation is impossible.
This notebook reconstructs the first-candidate recipe from the exact historical repo commit and then runs the current corrected re-evaluation contract.
This is recovery reconstruction, not iteration-2 learning.

In [ ]:
import os, sys, subprocess, json, shutil
from pathlib import Path
import torch
assert torch.cuda.is_available(), "GPU required"
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
subprocess.run([sys.executable,"-m","pip","install","-q","-U","transformers>=4.51,<5","peft>=0.15,<1","datasets>=3,<5","accelerate","bitsandbytes","sentencepiece","huggingface-hub"],check=True)


In [ ]:
OLD="865def7a98aa6b645b1950cfcd4ec9ffc37320ca"
TRAIN=Path("/content/tamdeshevle-train")
EVAL=Path("/content/tamdeshevle-eval")
for x in (TRAIN,EVAL):
    if x.exists(): shutil.rmtree(x)
subprocess.run(["git","clone","https://github.com/eneonstudio-dev/tamdeshevle.git",str(TRAIN)],check=True)
subprocess.run(["git","-C",str(TRAIN),"checkout",OLD],check=True)
subprocess.run(["git","clone","--depth","1","https://github.com/eneonstudio-dev/tamdeshevle.git",str(EVAL)],check=True)
print("train",subprocess.check_output(["git","-C",str(TRAIN),"rev-parse","HEAD"],text=True).strip())
print("eval",subprocess.check_output(["git","-C",str(EVAL),"rev-parse","HEAD"],text=True).strip())


In [ ]:
seed=Path("/content/bai-recovery-seed")
dataset=Path("/content/bai-recovery-dataset")
for x in (seed,dataset):
    if x.exists(): shutil.rmtree(x)
subprocess.run(["node",str(TRAIN/"teacher-lab/training/deterministic-seed.mjs"),str(seed)],cwd=TRAIN,check=True)
subprocess.run(["node",str(TRAIN/"teacher-lab/training/aggregate-gold.mjs"),str(dataset),str(seed/"gold.jsonl")],cwd=TRAIN,check=True)
print("train rows",sum(1 for x in (seed/"gold.jsonl").open() if x.strip()))
print("eval rows",sum(1 for x in (seed/"eval-gold.jsonl").open() if x.strip()))


In [ ]:
cand=Path("/content/rebuilt-first-bai-candidate")
if cand.exists(): shutil.rmtree(cand)
env={**os.environ,"CUDA_VISIBLE_DEVICES":"0"}
subprocess.run([
    sys.executable,str(TRAIN/"teacher-lab/training/train_student.py"),
    "--data",str(dataset/"sft.jsonl"),
    "--out",str(cand),
    "--config",str(TRAIN/"teacher-lab/training/student-v0.1.json")
],cwd=TRAIN,env=env,check=True)
adapter=cand/"adapter"
assert (adapter/"adapter_config.json").is_file()
print("adapter ready",adapter)


In [ ]:
out=Path("/content/bai-recovery-corrected-reeval")
if out.exists(): shutil.rmtree(out)
r=subprocess.run([
    sys.executable,str(EVAL/"teacher-lab/training/reevaluate_candidate.py"),
    "--eval-gold",str(seed/"eval-gold.jsonl"),
    "--candidate-adapter",str(adapter),
    "--out",str(out)
],cwd=EVAL,env=env)
print("reeval exit",r.returncode)
print((out/"reeval-manifest.json").read_text())


In [ ]:
pred=out/"candidate-predictions.jsonl"
assert pred.is_file()
fail=out/"failure-analysis"
subprocess.run([
    sys.executable,str(EVAL/"teacher-lab/training/analyze_candidate_failures.py"),
    "--eval-gold",str(seed/"eval-gold.jsonl"),
    "--predictions",str(pred),
    "--out-dir",str(fail)
],cwd=EVAL,check=True)
if (fail/"failure-summary.json").is_file():
    print((fail/"failure-summary.json").read_text())


In [ ]:
cmp=out/"comparison"
subprocess.run([
    sys.executable,str(EVAL/"teacher-lab/training/compare_candidate_runs.py"),
    "--eval-gold",str(seed/"eval-gold.jsonl"),
    "--run","corrected-baseline",str(out/"baseline-predictions.jsonl"),str(out/"metrics"/"baseline.json"),
    "--run","reconstructed-first-candidate",str(out/"candidate-predictions.jsonl"),str(out/"metrics"/"candidate.json"),
    "--out-dir",str(cmp)
],cwd=EVAL,check=True)
print((cmp/"candidate-comparison.md").read_text())


In [ ]:
prefix=Path("/content/bai_recovery_reeval")
subprocess.run([
    sys.executable,str(EVAL/"teacher-lab/training/package_reeval_artifact.py"),
    "--reeval-dir",str(out),
    "--eval-gold",str(seed/"eval-gold.jsonl"),
    "--candidate-adapter",str(adapter),
    "--out-prefix",str(prefix),
    "--source-ref","reconstructed-recipe-from:"+OLD
],cwd=EVAL,check=True)
handoff=Path(str(prefix)+"-handoff.json")
info=json.loads(handoff.read_text())
bundle=Path("/content/BAY_RECOVERY_RESULTS")
if bundle.exists(): shutil.rmtree(bundle)
bundle.mkdir()
for k in ("evidence_zip","adapter_zip"):
    src=Path(info[k]); shutil.copy2(src,bundle/src.name)
shutil.copy2(handoff,bundle/handoff.name)
final=shutil.make_archive("/content/BAY_RECOVERY_RESULTS","zip",bundle)
print("DONE")
print("Download:",final)
print(json.dumps(info,ensure_ascii=False,indent=2))


In [ ]:
from google.colab import files
final_zip = '/content/BAY_RECOVERY_RESULTS.zip'
print('Starting download:', final_zip)
files.download(final_zip)
